04-1-0 개념 설명용

# LangGraph Agent Memory

## State · Thread · Checkpoint · Checkpointer · Store

이 노트북은 LangGraph를 처음 배우는 학습자가 **Agent의 기억 구조를 직접 실행하면서 이해**하도록 구성한 실습 자료이다.

### 학습목표

실습 후 다음 내용을 설명할 수 있다.

1. LLM 자체 기억과 Agent 시스템 기억의 차이를 구분한다.
2. `State`와 `Thread`의 역할을 설명한다.
3. `Checkpointer`를 이용해 같은 Thread의 대화를 이어간다.
4. SQLite에 저장된 Thread 상태가 왜 **Thread 범위 기억**인지 설명한다.
5. `Store`를 이용해 서로 다른 Thread에서 정보를 공유한다.
6. 컴퓨터 저장장치 기준과 Agent Memory 기준을 구분한다.

### 실습 흐름

```text
기억 없는 함수
   ↓
State를 사용하는 Graph
   ↓
Checkpointer + thread_id
   ↓
SQLite 영구 저장
   ↓
Store 기반 Cross-thread Memory
```

> 이 실습은 외부 LLM API를 사용하지 않는다.  
> 규칙 기반 응답 함수를 사용하므로 API Key 없이 모든 핵심 개념을 확인할 수 있다.

## 0. 가장 먼저 구분할 두 가지 기준

| 구분 기준 | 단기 | 장기 |
|---|---|---|
| 컴퓨터 저장장치 기준 | RAM 등 휘발성 저장 | SSD·DB·파일 등 비휘발성 저장 |
| Agent Memory 기준 | 하나의 Thread 안에서 사용 | 여러 Thread에서 공유 |
| LangGraph 구성 | State + Checkpointer | Store |

핵심 문장은 다음과 같다.

> **DB에 오래 저장되어 있어도 하나의 Thread에서만 사용하면 Thread 범위 기억이다.**  
> **새로운 Thread에서도 재사용해야 할 정보는 Store에 별도로 저장한다.**

## 1. 실습 환경 준비

아래 셀은 필요한 패키지를 설치한다.

- `langgraph`: Graph, State, Checkpointer, Store 기능
- `langgraph-checkpoint-sqlite`: SQLite 기반 Checkpointer

설치 후 커널 재시작이 필요하다는 메시지가 나오면 Jupyter 메뉴에서 커널을 재시작한 뒤 다음 셀부터 실행한다.

In [82]:
# uv add langgraph langgraph-checkpoint-sqlite

In [83]:
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore

print("실습 환경 준비 완료")

실습 환경 준비 완료


## 2. 실습에 사용할 보조 함수

실습의 목적은 LLM 성능이 아니라 Memory 구조를 확인하는 것이다.

아래 `make_reply()` 함수는 이전 메시지에서 이름을 찾아 답변하는 간단한 규칙 기반 챗봇이다.

- 이전 메시지가 함께 전달되면 이름을 찾을 수 있다.
- 현재 질문만 전달되면 이름을 찾을 수 없다.

In [84]:
def get_message_text(message) -> str:
    """LangChain 메시지 객체 또는 딕셔너리에서 텍스트를 꺼낸다."""
    if hasattr(message, "content"):
        return str(message.content)
    return str(message.get("content", ""))


def find_name(messages) -> str | None:
    """대화 이력에서 '제 이름은 OOO입니다' 패턴을 찾는다."""
    pattern = re.compile(r"제 이름은\s*([가-힣A-Za-z0-9_-]+)")

    for message in reversed(messages):
        text = get_message_text(message)
        match = pattern.search(text)
        if match:
            return match.group(1)

    return None


def make_reply(messages) -> str:
    """현재까지 전달된 메시지를 보고 규칙 기반으로 답변한다."""
    latest = get_message_text(messages[-1])

    if "제 이름은" in latest:
        name = find_name(messages)
        return f"확인했습니다. 이름은 {name}입니다."

    if "이름" in latest and ("뭐" in latest or "무엇" in latest):
        name = find_name(messages[:-1])
        if name:
            return f"이전에 알려주신 이름은 {name}입니다."
        return "현재 전달된 정보만으로는 이름을 알 수 없습니다."

    return f"현재 메시지를 확인했습니다: {latest}"

## 3. 실습 1: 기억이 없는 함수

먼저 과거 대화를 저장하지 않는 일반 함수를 실행한다.

각 호출에는 현재 메시지 하나만 전달되므로 두 번째 호출에서 이름을 기억할 수 없다.

In [85]:
def memoryless_chat(user_message: str) -> str:
    current_messages = [
        {"role": "user", "content": user_message}
    ]
    return make_reply(current_messages)


print(memoryless_chat("제 이름은 joy입니다."))
print(memoryless_chat("제 이름은 무엇인가요?"))

확인했습니다. 이름은 joy입니다입니다.
확인했습니다. 이름은 무엇인가요입니다.


### 예상 결과

```text
확인했습니다. 이름은 joy입니다.
현재 전달된 정보만으로는 이름을 알 수 없습니다.
```

### 관찰 포인트

첫 번째 호출과 두 번째 호출은 서로 독립적이다.  
모델이나 함수가 기억하는 것이 아니라 **애플리케이션이 과거 정보를 다시 제공해야 한다.**

## 4. 실습 2: LangGraph State 사용

`MessagesState`는 LangGraph가 제공하는 기본 State이다.

```text
MessagesState
└─ messages: 사용자와 AI의 메시지 목록
```

Node는 State를 입력받고, 새 메시지를 반환한다.

In [86]:
def chatbot_node(state: MessagesState):
    reply = make_reply(state["messages"])

    # 새 메시지만 반환한다.
    # MessagesState의 reducer가 기존 메시지 뒤에 추가한다.
    return {
        "messages": [
            {"role": "assistant", "content": reply}
        ]
    }


builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph_without_memory = builder.compile()

print("기억 기능이 없는 Graph 생성 완료")

기억 기능이 없는 Graph 생성 완료


같은 Graph를 사용해도 Checkpointer가 없으면 각 `invoke()` 호출의 State는 이어지지 않는다.

In [87]:
result_1 = graph_without_memory.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 joy입니다."}]}
)
    
result_2 = graph_without_memory.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 무엇인가요?"}]}
)

print("첫 번째 답변:", result_1["messages"][-1].content)
print("두 번째 답변:", result_2["messages"][-1].content)

첫 번째 답변: 확인했습니다. 이름은 joy입니다입니다.
두 번째 답변: 확인했습니다. 이름은 무엇인가요입니다.


### 핵심 정리

`State`는 Graph 실행 중 Node들이 공유하는 작업 공간이다.

그러나 **State를 다음 실행까지 이어가려면 Checkpointer가 필요하다.**

## 5. 실습 3: InMemorySaver로 같은 Thread 이어가기

이제 Checkpointer를 연결한다.

```text
State
  ↓ 저장
Checkpointer
  ↓ thread_id로 구분
Thread별 Checkpoint
```

`InMemorySaver`는 RAM에 Checkpoint를 저장한다.  
프로그램이 실행 중인 동안에는 대화를 이어갈 수 있지만 커널을 재시작하면 사라진다.

In [88]:
memory_checkpointer = InMemorySaver()

graph_with_memory = builder.compile(
    checkpointer=memory_checkpointer
)

thread_a = {
    "configurable": {
        "thread_id": "thread-a"
    }
}

print("InMemorySaver 연결 완료")

InMemorySaver 연결 완료


In [89]:
# 첫 번째 Run
first = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 joy입니다."}]},
    config=thread_a,
)

# 두 번째 Run: 같은 thread_id 사용
second = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 무엇인가요?"}]},
    config=thread_a,
)

print(first["messages"][-1].content)
print(second["messages"][-1].content)

확인했습니다. 이름은 joy입니다입니다.
확인했습니다. 이름은 무엇인가요입니다.


### 예상 결과

```text
확인했습니다. 이름은 joy입니다.
이전에 알려주신 이름은 joy입니다.
```

같은 `thread_id`를 사용했기 때문에 Checkpointer가 이전 State를 불러와 새 메시지와 연결한다.

### 저장된 State 확인

`get_state()`를 사용하면 해당 Thread의 최신 State를 확인할 수 있다.

In [90]:
snapshot = graph_with_memory.get_state(thread_a)

print("다음 실행 Node:", snapshot.next)
print("누적 메시지 수:", len(snapshot.values["messages"]))
print()

for message in snapshot.values["messages"]:
    print(f"{message.type:>5}: {message.content}")

다음 실행 Node: ()
누적 메시지 수: 4

human: 제 이름은 joy입니다.
   ai: 확인했습니다. 이름은 joy입니다입니다.
human: 제 이름은 무엇인가요?
   ai: 확인했습니다. 이름은 무엇인가요입니다.


### Checkpoint 이력 확인

하나의 Thread 안에는 실행 단계별 Checkpoint가 여러 개 만들어진다.

In [91]:
history = list(graph_with_memory.get_state_history(thread_a))

print("Checkpoint 개수:", len(history))

for index, item in enumerate(history[:5], start=1):
    checkpoint_id = item.config["configurable"].get("checkpoint_id")
    message_count = len(item.values.get("messages", []))
    print(
        f"{index}. checkpoint_id={checkpoint_id}, "
        f"messages={message_count}, next={item.next}"
    )

Checkpoint 개수: 6
1. checkpoint_id=1f18df18-4d5e-6f23-8004-fa0c2f460f6e, messages=4, next=()
2. checkpoint_id=1f18df18-4d5c-6816-8003-5b8e3562fb1b, messages=3, next=('chatbot',)
3. checkpoint_id=1f18df18-4d5c-6815-8002-726a694e465a, messages=2, next=('__start__',)
4. checkpoint_id=1f18df18-4d54-6870-8001-bffc1fcb8f5a, messages=2, next=()
5. checkpoint_id=1f18df18-4d4a-6b8b-8000-83f171b40de8, messages=1, next=('chatbot',)


## 6. 실습 4: 다른 Thread에서는 기억이 분리되는가

새로운 `thread_id`는 새로운 채팅 페이지와 비슷하다.

In [92]:
thread_b = {
    "configurable": {
        "thread_id": "thread-b"
    }
}

new_thread_result = graph_with_memory.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 무엇인가요?"}]},
    config=thread_b,
)

print(new_thread_result["messages"][-1].content)

확인했습니다. 이름은 무엇인가요입니다.


### 관찰 포인트

`thread-a`에는 이름이 저장되어 있지만 `thread-b`는 별도의 State를 가진다.

```text
thread-a: joy라는 이름을 알고 있음
thread-b: 이전 이름 정보가 없음
```

따라서 Checkpointer는 **Thread별 상태를 저장**하지만 다른 Thread에 자동으로 공유하지 않는다.

## 7. 실습 5: SQLite Checkpointer로 프로그램 종료 후에도 유지하기

이번에는 Checkpoint를 SQLite 파일에 저장한다.

```text
InMemorySaver
└─ RAM 저장: 커널 종료 시 사라짐

SqliteSaver
└─ SQLite 파일 저장: 커널 종료 후에도 유지됨
```

중요한 점은 SQLite에 오래 저장되더라도, 정보의 활용 범위가 하나의 Thread이면 LangGraph에서는 **Thread 범위 기억**이라는 점이다.

In [93]:
DB_PATH = Path("langgraph_memory_lab.sqlite")

# 셀을 반복 실행할 때 기존 연결과 파일을 정리한다.
if "sqlite_connection" in globals():
    try:
        sqlite_connection.close()
    except Exception:
        pass

if "sqlite_connection_2" in globals():
    try:
        sqlite_connection_2.close()
    except Exception:
        pass

if DB_PATH.exists():
    DB_PATH.unlink()

sqlite_connection = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

sqlite_checkpointer = SqliteSaver(sqlite_connection)
sqlite_checkpointer.setup()

graph_sqlite = builder.compile(
    checkpointer=sqlite_checkpointer
)

sqlite_thread = {
    "configurable": {
        "thread_id": "sqlite-thread-001"
    }
}

print("SQLite 파일:", DB_PATH.resolve())

SQLite 파일: C:\Users\magpi\agentic_ai_lab_202607\track1_core\langgraph_memory_lab.sqlite


In [94]:
graph_sqlite.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 joy입니다."}]},
    config=sqlite_thread,
)

saved_state = graph_sqlite.get_state(sqlite_thread)

print("저장된 메시지 수:", len(saved_state.values["messages"]))
print("SQLite 파일 존재 여부:", DB_PATH.exists())
print("SQLite 파일 크기:", DB_PATH.stat().st_size, "bytes")

저장된 메시지 수: 2
SQLite 파일 존재 여부: True
SQLite 파일 크기: 4096 bytes


### SQLite 내부 테이블 확인

내부 테이블은 LangGraph가 관리한다.  
구조를 관찰할 수는 있지만 애플리케이션에서 직접 수정하지 않는 것이 좋다.

In [95]:
tables = sqlite_connection.execute(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """
).fetchall()

print("생성된 테이블")
for table_name, in tables:
    print("-", table_name)

생성된 테이블
- checkpoints
- writes


### 프로그램 재시작 상황 모의 실험

기존 SQLite 연결을 닫고, 같은 파일을 다시 연결한다.  
이는 오늘 프로그램을 종료한 뒤 내일 다시 실행하는 상황과 유사하다.

In [96]:
# 기존 프로그램 종료를 모의한다.
sqlite_connection.commit()
sqlite_connection.close()

# 새로운 프로그램 실행을 모의한다.
sqlite_connection_2 = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

sqlite_checkpointer_2 = SqliteSaver(sqlite_connection_2)
sqlite_checkpointer_2.setup()

graph_sqlite_reopened = builder.compile(
    checkpointer=sqlite_checkpointer_2
)

reopened_result = graph_sqlite_reopened.invoke(
    {"messages": [{"role": "user", "content": "제 이름은 무엇인가요?"}]},
    config=sqlite_thread,  # 동일한 thread_id
)

print(reopened_result["messages"][-1].content)

확인했습니다. 이름은 무엇인가요입니다.


### 핵심 해석

위 데이터는 SQLite에 비휘발성으로 저장되었다.  
그러나 `sqlite-thread-001`이라는 하나의 Thread 안에서만 사용된다.

따라서 두 문장이 동시에 성립한다.

```text
컴퓨터 저장장치 기준: 영구 저장
Agent Memory 기준: Thread 범위 기억
```

## 8. 실습 6: Store로 서로 다른 Thread에서 정보 공유하기

Store는 Graph State 밖에 있는 애플리케이션 정의 정보를 저장한다.

이번 실습에서는 사용자별 설명 형식을 Store에 저장한다.

```text
Thread A에서 선호 형식 저장
          ↓
Store: user-001/profile
          ↓
Thread B에서 같은 사용자 정보 조회
```

이 실습에서는 구조를 쉽게 확인하기 위해 `InMemoryStore`를 사용한다.

- 활용 범위: 여러 Thread
- 물리적 저장 위치: RAM
- 운영 환경: PostgreSQL, MongoDB, Redis 등의 영구 Store 권장

In [97]:
@dataclass
class UserContext:
    user_id: str


def extract_preference(text: str) -> str | None:
    """'선호하는 설명 형식은 OOO입니다' 문장에서 OOO를 추출한다."""
    pattern = re.compile(
        r"선호하는 설명 형식은\s*(.+?)(?:입니다|이다|예요|이에요|\.|$)"
    )
    match = pattern.search(text)
    return match.group(1).strip() if match else None


def store_memory_node(
    state: MessagesState,
    runtime: Runtime[UserContext],
):
    latest = get_message_text(state["messages"][-1])
    user_id = runtime.context.user_id
    namespace = (user_id, "profile")

    # 1. 사용자가 기억할 정보를 말한 경우 Store에 저장한다.
    preference = extract_preference(latest)
    if preference:
        runtime.store.put(
            namespace,
            "explanation_style",
            {"value": preference},
        )

        reply = (
            f"장기기억에 저장했습니다. "
            f"선호하는 설명 형식은 '{preference}'입니다."
        )
        return {"messages": [{"role": "assistant", "content": reply}]}

    # 2. 사용자가 선호 형식을 질문한 경우 Store에서 조회한다.
    if "선호" in latest and ("뭐" in latest or "무엇" in latest):
        item = runtime.store.get(
            namespace,
            "explanation_style",
        )

        if item:
            reply = (
                "Store에서 조회한 선호 설명 형식은 "
                f"'{item.value['value']}'입니다."
            )
        else:
            reply = "Store에 저장된 선호 설명 형식이 없습니다."

        return {"messages": [{"role": "assistant", "content": reply}]}

    # 3. 그 외에는 앞서 만든 Thread 범위 응답을 사용한다.
    return {
        "messages": [
            {"role": "assistant", "content": make_reply(state["messages"])}
        ]
    }

In [98]:
shared_store = InMemoryStore()
thread_checkpointer = InMemorySaver()

store_builder = StateGraph(
    MessagesState,
    context_schema=UserContext,
)

store_builder.add_node("memory_agent", store_memory_node)
store_builder.add_edge(START, "memory_agent")
store_builder.add_edge("memory_agent", END)

graph_with_store = store_builder.compile(
    checkpointer=thread_checkpointer,
    store=shared_store,
)

print("Checkpointer와 Store가 함께 연결된 Graph 생성 완료")

Checkpointer와 Store가 함께 연결된 Graph 생성 완료


### Thread A에서 사용자 선호 저장

In [99]:
store_thread_a = {
    "configurable": {
        "thread_id": "user-001-thread-a"
    }
}

save_result = graph_with_store.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "제가 선호하는 설명 형식은 표 중심입니다."
            }
        ]
    },
    config=store_thread_a,
    context=UserContext(user_id="user-001"),
)

print(save_result["messages"][-1].content)

장기기억에 저장했습니다. 선호하는 설명 형식은 '표 중심'입니다.


### 새로운 Thread B에서 같은 사용자 정보 조회

`thread_id`는 달라졌지만 `user_id`가 같다.

In [100]:
store_thread_b = {
    "configurable": {
        "thread_id": "user-001-thread-b"
    }
}

load_result = graph_with_store.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "제가 선호하는 설명 형식은 무엇인가요?"
            }
        ]
    },
    config=store_thread_b,
    context=UserContext(user_id="user-001"),
)

print(load_result["messages"][-1].content)

장기기억에 저장했습니다. 선호하는 설명 형식은 '무엇인가요?'입니다.


### 예상 결과

```text
Store에서 조회한 선호 설명 형식은 '표 중심'입니다.
```

Thread A와 Thread B는 서로 다른 State를 가진다.  
그러나 같은 `user_id`의 Store namespace를 사용했기 때문에 정보를 공유할 수 있다.

### 다른 사용자는 같은 정보를 볼 수 없는가

이번에는 `user_id`를 변경한다.

In [101]:
other_user_thread = {
    "configurable": {
        "thread_id": "user-002-thread-a"
    }
}

other_user_result = graph_with_store.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "제가 선호하는 설명 형식은 무엇인가요?"
            }
        ]
    },
    config=other_user_thread,
    context=UserContext(user_id="user-002"),
)

print(other_user_result["messages"][-1].content)

장기기억에 저장했습니다. 선호하는 설명 형식은 '무엇인가요?'입니다.


### Store 내용 직접 확인

Store는 namespace와 key를 사용해 데이터를 구분한다.

In [102]:
items = shared_store.search(
    ("user-001", "profile"),
    limit=100,
)

for item in items:
    print("namespace:", item.namespace)
    print("key      :", item.key)
    print("value    :", item.value)
    print()

namespace: ('user-001', 'profile')
key      : explanation_style
value    : {'value': '무엇인가요?'}



## 9. Checkpointer와 Store 비교 실험 정리

| 실험 | 저장장치 | 활용 범위 | LangGraph Memory 분류 |
|---|---|---|---|
| `InMemorySaver` | RAM | 동일 Thread | Thread 범위 기억 |
| `SqliteSaver` | SQLite 파일 | 동일 Thread | Thread 범위 기억 |
| `InMemoryStore` | RAM | 여러 Thread | Cross-thread Memory |
| `PostgresStore` 등 | DB | 여러 Thread | Cross-thread Memory |

### 가장 중요한 관찰

1. SQLite에 저장했다고 자동으로 Cross-thread Memory가 되지 않는다.
2. InMemoryStore가 RAM에 있어도 여러 Thread에서 공유하면 논리적으로 Cross-thread Memory이다.
3. 저장 기간과 활용 범위는 서로 다른 축이다.

## 10. 직접 해보기

### 과제 1. Thread 분리 확인

아래 코드의 `thread_id`를 변경한 뒤 이름을 기억하는지 확인한다.

```python
config = {
    "configurable": {
        "thread_id": "새로운-thread-id"
    }
}
```

### 과제 2. Store에 회사 정보 저장

다음 문장을 처리하도록 `store_memory_node()`를 확장한다.

```text
우리 회사의 AI 서비스 유형은 채용 AI입니다.
```

권장 Store 구조는 다음과 같다.

```python
namespace = (user_id, "organization")
key = "service_type"
value = {"value": "채용 AI"}
```

### 과제 3. 같은 사용자, 다른 Thread

- Thread C를 새로 만든다.
- `user_id="user-001"`을 유지한다.
- Store에 저장한 선호 설명 형식을 조회한다.

### 과제 4. 다른 사용자 격리

- `user_id="user-003"`으로 실행한다.
- `user-001`의 Store 정보가 조회되지 않는지 확인한다.

## 11. 확인 문제

1. `State`와 `Checkpoint`의 차이는 무엇인가?
2. `thread_id`는 무엇을 구분하는가?
3. SQLite에 한 달 동안 저장된 대화가 Thread 범위 기억일 수 있는 이유는 무엇인가?
4. Store는 Checkpoint 전체를 자동 복사하는가?
5. 서로 다른 Thread에서 같은 사용자 정보를 공유하려면 무엇이 필요한가?
6. `user_id`와 `thread_id`는 어떤 차이가 있는가?

## 12. 정답 및 해설

1. **State**는 현재 Graph가 처리하는 작업 데이터이고, **Checkpoint**는 특정 시점에 저장된 State 스냅샷이다.
2. `thread_id`는 하나의 연속된 대화나 작업을 구분한다.
3. 저장 기간은 길지만 활용 범위가 하나의 Thread에 한정되어 있기 때문이다.
4. 아니다. 필요한 정보를 직접 추출해 `store.put()`으로 저장해야 한다.
5. 사용자 또는 조직을 식별하는 namespace와 Store가 필요하다.
6. `user_id`는 사용자를 식별하고, `thread_id`는 해당 사용자의 특정 대화나 작업을 식별한다.

## 13. 최종 정리

```text
State
= 현재 실행 중인 작업 정보

Checkpoint
= 특정 시점에 저장된 State

Checkpointer
= Thread별 State 저장·복원 장치

Thread
= 시간과 관계없이 이어지는 하나의 대화·작업 단위

Store
= 여러 Thread에서 사용할 애플리케이션 정보를 저장하는 장치
```

### 강의에서 사용할 핵심 문장

> **Agent Memory의 단기·장기는 저장 기간보다 활용 범위로 구분한다.**

> **Checkpointer는 하나의 Thread를 이어가고, Store는 여러 Thread에서 정보를 공유하게 한다.**

## 14. 실습 종료 후 자원 정리

SQLite 연결을 닫는다.

In [103]:
if "sqlite_connection_2" in globals():
    sqlite_connection_2.commit()
    sqlite_connection_2.close()

print("SQLite 연결 종료 완료")

SQLite 연결 종료 완료


## 참고 자료

본 실습은 LangGraph 공식 문서의 현재 구조를 기준으로 작성했다.

- Persistence  
  https://docs.langchain.com/oss/python/langgraph/persistence

- Memory  
  https://docs.langchain.com/oss/python/langgraph/add-memory

- Stores  
  https://docs.langchain.com/oss/python/langgraph/stores

- SqliteSaver API Reference  
  https://reference.langchain.com/python/langgraph.checkpoint.sqlite/SqliteSaver